In [ ]:
from pathlib import Path
import fitz
import pytesseract
import os
import shutil
from PIL import Image
import cv2


_tesseract = os.environ.get("TESSERACT_CMD") or shutil.which("tesseract")
if not _tesseract:
    for _candidate in (r"C:\Program Files\Tesseract-OCR\tesseract.exe", r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe"):
        if Path(_candidate).is_file():
            _tesseract = _candidate
            break
if _tesseract:
    pytesseract.pytesseract.tesseract_cmd = _tesseract

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

def extract_text_from_image(image_path):
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Image not found or unreadable: {image_path}")
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return pytesseract.image_to_string(gray, config="--oem 3 --psm 6")

def extract_text_from_pdf(pdf_path):
    pages = []
    with fitz.open(pdf_path) as document:
        for page in document:
            page_text = page.get_text("text").strip()
            if not page_text:
                pix = page.get_pixmap(dpi=250, alpha=False)
                image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                page_text = pytesseract.image_to_string(image)
            pages.append(page_text)
    return "\n".join(pages)

sample = PROJECT_ROOT / "data" / "prescriptions" / "prescription_01.txt"
text = sample.read_text(encoding="utf-8")
print("Loaded current text sample (OCR will be used after image/PDF samples are added):")
print(text)
